# 00 - Setup and configuration

**Stage question.** How is a single experimental run identified, and can that identifier be trusted to be stable where it must be and sensitive where it must be?

Every fit in CODA is keyed by a hash of its configuration. Results are written under that key and never overwritten. Two properties have to hold, and both are load-bearing for claims made much later:

1. **Stability.** Two runs of the same fit must produce the same identifier, even when one ran on this machine and the other on a cloud runtime. Otherwise the same analysis reproduced elsewhere looks like a different experiment.
2. **Sensitivity.** Two runs differing in *anything* that affects the result must produce different identifiers. Otherwise two experimental conditions could silently write into the same place, and the second would overwrite the first without warning.

This notebook exercises `src/config.py` directly. It defines no logic of its own.

**Terminology used below.**

- **Condition** - one of the four information settings the main experiment compares. They differ only in what the model is allowed to see.
- **Donor scheme** - a control in which the partner's turn is replaced by a different turn, to test whether a result depends on the genuine pairing.
- **Partner mode** - how the partner's acoustic pathway is fed. Changing it changes the information available and never the number of parameters.
- **Block** - a group of related delivery measurements. Statistical testing happens at block level, not per individual measurement.

In [1]:
# Runtime setup — identical in every notebook.
# Colab: mounts Google Drive and uses MyDrive/coda (a copy of the repository) as
#        the project root, then installs the pinned measurement stack.
# Local: uses the repository checkout.
# Every path used below is relative to that root, so Drive and the repository
# have exactly the same layout and outputs can be copied back unchanged.
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    ROOT = Path(os.environ.get('CODA_ROOT', '/content/drive/MyDrive/coda'))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(ROOT / 'requirements-colab.txt')], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

os.environ['CODA_ROOT'] = str(ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.runtime import runtime_report
REPORT = runtime_report()

Mounted at /content/drive
runtime        colab   python 3.13.16   cpus 2
memory         12.4 GB free of 13.6 GB
gpu            none (not needed for this stage unless stated)
project disk   13.4 GB free   root = /content/drive/MyDrive/coda
scratch disk   93.4 GB free   (audio lives here briefly)
code           fingerprint 51ce4acb032a1d84
artifacts      manifest=True, metadata=True, transcripts=False, turns=False, audio_pass_sessions=0, channel_qc=False


**Runtime check.** The cell above prints free disk on the project root (Drive on Colab) and on the scratch disk where audio briefly lives, memory, CPU count, GPU, the code fingerprint, and which pipeline artifacts already exist. The code fingerprint identifies the exact source and configuration that ran; it is written into every run record and is how a result produced on Drive is matched back to the repository.

In [2]:
import json
from dataclasses import replace

from src.config import (
    TARGET_BLOCKS, TARGET_DIMENSIONS, N_TARGET_DIMENSIONS,
    RunConfig, DataConfig, NeuralConfig,
    environment_record, assert_same_capacity,
)
from src.paths import paths

## 0. Where everything lives

Every notebook runs on Colab from a copy of this repository on Google Drive, at `MyDrive/coda/`. All paths in the code are relative to one project root, resolved in `src/paths.py`, so the folder layout on Drive and in the repository is identical by construction. A result produced on Drive is copied back to the same relative path in the repository.

| Folder | Holds | In git |
|---|---|---|
| `src/`, `configs/`, `tests/`, `notebooks/` | code, configuration, integrity suite, the stage notebooks | yes |
| `manifests/` | the download manifest: exactly which files form the sample | yes |
| `results/` | tables, logs, gate verdicts and run records produced by the notebooks | yes |
| `figures/` | every figure the notebooks save | yes |
| `docs/` | research overview, gate verdicts | yes |
| `data/raw/` | corpus metadata, transcripts, voice-activity segments | no (corpus-derived) |
| `data/processed/` | turn tables, channel QC, pitch and intensity tracks | no (corpus-derived, regenerable) |

Audio is never stored in any of these. It is streamed to the Colab runtime's local disk, measured, and deleted within seconds.

**Bringing results back.** After a Colab session, copy `results/`, `figures/`, `manifests/` and any notebook you ran (with its outputs) from Drive into the repository at the same paths. `data/` may also be copied if a local analysis needs it; git ignores it. Every run record carries a code fingerprint. The next cell prints the current one, and it must match the fingerprint in the records being copied back. A mismatch means the Drive copy of the code differs from the repository.

In [3]:
P = paths()
from src.runtime import code_fingerprint
print('project root     ', P.root)
print('code fingerprint ', code_fingerprint())
for name in ('configs', 'manifests', 'results', 'figures', 'raw_metadata', 'raw_transcript',
             'raw_vad', 'processed', 'audio_scratch'):
    d = getattr(P, name)
    print(f'{name:<16} {d}   exists={d.exists()}')

project root      /content/drive/MyDrive/coda
code fingerprint  51ce4acb032a1d84
configs          /content/drive/MyDrive/coda/configs   exists=True
manifests        /content/drive/MyDrive/coda/manifests   exists=True
results          /content/drive/MyDrive/coda/results   exists=True
figures          /content/drive/MyDrive/coda/figures   exists=True
raw_metadata     /content/drive/MyDrive/coda/data/raw/metadata   exists=True
raw_transcript   /content/drive/MyDrive/coda/data/raw/transcript   exists=False
raw_vad          /content/drive/MyDrive/coda/data/raw/vad   exists=False
processed        /content/drive/MyDrive/coda/data/processed   exists=False
audio_scratch    /content/coda_scratch/audio   exists=True


## 1. The target being predicted

The quantity this study predicts is the *delivery* of a reply: how it was said, not what was said. It is measured as eight numbers per turn, grouped into four blocks.

In [4]:
for block, dims in TARGET_BLOCKS.items():
    print(f'{block:<20} {len(dims)}  ' + ', '.join(dims))
print()
print(f'{N_TARGET_DIMENSIONS} dimensions in {len(TARGET_BLOCKS)} blocks')

pitch_level_span     2  mean_log_f0, log_f0_range_p95_p5
pitch_boundary       1  log_f0_slope_final_300ms
energy               2  mean_intensity, intensity_range
timing               3  articulation_rate, pause_ratio, longest_internal_pause

8 dimensions in 4 blocks


**What this shows.** Eight measurements in four groups: where the voice sits in pitch and how far it ranges; how the pitch moves at the very end of the turn; how loud it is and how much the loudness varies; and three aspects of timing.

**Why it is grouped.** Statistical testing is done per block, with each block weighted equally, giving four tests rather than eight. This matters for two reasons. Testing eight correlated measurements separately would inflate the chance of a spurious finding. And weighting by block stops timing, which has three measurements, from dominating the pitch boundary, which has one, purely by count.

**A caution worth recording.** Earlier project drafts describe this target as *nine* dimensions in their prose while listing eight in the table. No draft names a ninth. The enumeration is the definition, and the correction is recorded in the preregistration before any confirmatory result. The relevant consequence is narrow: it changes how wide the prediction head is, and nothing about the hypotheses, the tests, or the controls.

## 2. Run identifiers

A configuration describes one fit completely. Its hash is that fit's name.

In [5]:
primary = RunConfig(experiment='j1', condition='plus_partner_delivery')
print('primary comparison :', primary.run_id())
print('results directory  :', primary.result_dir())

primary comparison : a60bfff9ea9e92ce
results directory  : results/j1__a60bfff9ea9e92ce


### 2.1 Stability

The same configuration must always resolve to the same identifier, and free-text notes must not disturb it.

In [6]:
same = RunConfig(experiment='j1', condition='plus_partner_delivery')
noted = replace(primary, notes='rerun after a preempted cloud session')

print('rebuilt identically :', same.run_id() == primary.run_id())
print('notes added         :', noted.run_id() == primary.run_id())

rebuilt identically : True
notes added         : True


**What this shows.** Rebuilding the configuration from scratch reproduces the identifier, and annotating a run does not change it.

**Why it matters.** Part of this project's computation runs on a cloud runtime and part runs locally. If the environment leaked into the identifier, the same fit would be filed under two names and its provenance would fracture. The environment is therefore *recorded* alongside each run but deliberately excluded from the hash.

### 2.2 Sensitivity

Anything that could change the result must change the identifier. The four experimental conditions are the case that matters most.

In [7]:
conds = ['Z_content', 'plus_partner_text', 'plus_partner_delivery', 'plus_self_previous']
for c in conds:
    print(f'{c:<24} ' + replace(primary, condition=c).run_id())
print()
print('all distinct:', len({replace(primary, condition=c).run_id() for c in conds}) == 4)

Z_content                01ca978c3f413a4e
plus_partner_text        ffe64b78051700d2
plus_partner_delivery    a60bfff9ea9e92ce
plus_self_previous       3ca15608149aa5bb

all distinct: True


In [8]:
# A change buried inside a nested configuration must still surface.
deeper = replace(primary, data=DataConfig(split_seed=7))
print('changed split seed ->', deeper.run_id(),
      '| differs:', deeper.run_id() != primary.run_id())

changed split seed -> 07df23e9d83f6eee | differs: True


**What this shows.** Each condition gets its own identifier, and a change buried several levels down in the configuration still produces a different one.

**Why it matters.** The central comparison in this study is between conditions that are deliberately similar, differing only in what information the model may use. They will be run repeatedly, under several control schemes and several random seeds. A collision between any two would mean one silently overwrote another's results, and the error would be invisible in the output. The sensitivity check is what rules that out.

**What this does not establish.** That the hash is cryptographically meaningful, or that the configuration captures everything relevant. It captures what has been declared. A change made outside the configuration, by editing code, is not visible here, which is why the commit is recorded separately.

## 3. Equal capacity across conditions

The neural part of this study compares conditions that see different information. For that comparison to say anything about *information*, everything else must be held fixed: the same architecture, the same number of trainable parameters, the same optimiser and training budget.

This is the single easiest way for a study like this to fool itself. A model given more input and more capacity will usually do better, and the improvement says nothing about whether the extra input carried useful information. So capacity equality is asserted in code rather than assumed.

In [9]:
shared = NeuralConfig()
modes = ('real', 'donor', 'null_token', 'delivery_vector_interpretable')
runs = [RunConfig(experiment='neural', condition='plus_partner_delivery',
                  partner_mode=m, neural=shared) for m in modes]

assert_same_capacity(runs)
print('capacity identical across all four partner modes')
for r in runs:
    print(f'  {r.partner_mode:<32} ' + r.run_id())

capacity identical across all four partner modes
  real                             dc32f5577012747c
  donor                            b8b5c180477907b4
  null_token                       2b97b438c984d513
  delivery_vector_interpretable    2189d3c5f0396450


In [10]:
# And it must fail loudly when an architecture actually differs.
tampered = [runs[0], replace(runs[1], neural=NeuralConfig(d_model=256))]
try:
    assert_same_capacity(tampered)
    print('NOT CAUGHT - this would be a defect')
except AssertionError as err:
    print('caught as expected:', err)

caught as expected: Neural configuration differs across conditions, which breaks capacity control: neural/plus_partner_delivery


**What this shows.** Four different ways of feeding the partner's audio share one architecture and one parameter count, yet each is filed under its own identifier. When a genuine architectural difference is introduced, the check refuses it.

**Why it matters.** The comparison this study relies on is *same model, same capacity, same training, different information available*. Published work in this area often compares systems that differ in modality, size and optimisation all at once, which is why those results establish that acoustic context helps without establishing how much it contributes. The assertion above is what keeps this study out of that trap.

**Note on the ablation.** In the condition where the partner's audio is withheld, the pathway is not deleted. It receives a learned placeholder sequence instead. Deleting it would shrink the model and reintroduce the very confound being controlled for.

## 4. Execution environment

Every run records where it happened and which code produced it.

In [11]:
print(json.dumps(environment_record(), indent=2))

{
  "runtime": "colab",
  "git_commit": "566d29752af99aac4901dae0469c7887f82fe506",
  "code_fingerprint": "51ce4acb032a1d84",
  "python": "3.13.16",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39"
}


**What this shows.** The runtime, the commit (when run from a git checkout), the code fingerprint and the interpreter version.

**Why it matters.** The notebooks run on Colab from a Drive copy of the repository, which has no git history. There the code fingerprint identifies the exact source and configuration that produced a result, and it is checked against the repository when results are copied back. The pitch-tracking library is the sharpest reason to pin versions: if its defaults changed between runs, every target value would shift, and nothing downstream would announce the problem.

A `None` commit is expected on Colab; the fingerprint takes its place.

## 5. Where results go

One directory per run, named by experiment and identifier.

In [12]:
for c in conds:
    print(replace(primary, condition=c).result_dir())

results/j1__01ca978c3f413a4e
results/j1__ffe64b78051700d2
results/j1__a60bfff9ea9e92ce
results/j1__3ca15608149aa5bb


**What this shows.** Each condition writes to its own directory, so results accumulate rather than overwrite.

**Why it matters.** Every number that eventually appears in a figure or table has to be traceable to the run that produced it. Keeping run directories separate and append-only is the mechanism. It also means an experiment can be re-run after a change without destroying the earlier evidence, which is what allows a change of approach to be documented rather than silently erased.

## 6. Summary

The run-identifier scheme behaves as required: stable across environments and across incidental annotation, sensitive to every declared change including nested ones, and sufficient to keep the four experimental conditions separate on disk.

Capacity equality across the partner modes is asserted in code and fails loudly when violated.

**Limitations of what was checked here.** Only *declared* configuration is hashed. Changes made by editing code are not visible to it, which is why the commit is recorded alongside each run and why library versions are pinned rather than left to resolve. Every measurement is produced on one runtime, Colab, with the measurement stack pinned in `requirements-colab.txt`, so there is no second environment whose library defaults could disagree.

**Next stage.** `01_corpus_frame_and_manifest`: the sampling frame, the seeded selection and the download manifest.